# Week 8 — Retrieval Experiments (sandbox)

Calibrates every retrieval decision **before** any of it moves into `src/rag/week8/`, the same way the Week 7 notebook calibrated parsing and chunking.

**Questions this notebook answers**
1. What do embeddings / cosine similarity look like with the chosen model, including Arabic?
2. Do our chunks fit the model's token limit (no silent truncation)?
3. Embed `text_only` or `section_header + text`? (decided by numbers, not by taste)
4. What `top_k` and what `RAG_MIN_SCORE` threshold separate answerable from unanswerable questions?
5. How does ChromaDB actually behave with our metadata (NULLs, filters, cosine, re-upsert, sync)?

**Inputs:** `data/db/local_storage.sqlite` (`rag_documents` / `rag_chunks` from Week 7, after the *replace* fix) · **Writes:** `data/eval/week8/eval_questions.json` only if it does not exist yet. Chroma experiments use a temp folder, never `data/db/chroma/`.

**Requires:** `pip install sentence-transformers chromadb` (first run downloads the model, ~470 MB).

## 0. Setup

In [1]:
import fnmatch
import json
import os
import shutil
import sqlite3
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("Project root not found (expected a folder containing data/ and src/).")


BASE_DIR = find_project_root()
DB_PATH = BASE_DIR / "data" / "db" / "local_storage.sqlite"
EVAL_PATH = BASE_DIR / "data" / "eval" / "week8" / "eval_questions.json"
MODEL_NAME = os.getenv("EMBEDDING_MODEL_NAME", "paraphrase-multilingual-MiniLM-L12-v2")

print("Project root:", BASE_DIR)
print("DB exists   :", DB_PATH.exists())
print("Model       :", MODEL_NAME)

Project root: E:\DigitalFutureProjects\TrainingProjects\standalone-data-rag-assistant
DB exists   : True
Model       : paraphrase-multilingual-MiniLM-L12-v2


## 1. Source of truth: what is in `rag_chunks`?

SQLite stays the source of truth; Chroma will only be an index derived from it. First a sanity check that the Week 7 *replace* fix worked (no duplicated documents).

In [2]:
with sqlite3.connect(DB_PATH) as conn:
    docs_df = pd.read_sql_query("SELECT * FROM rag_documents ORDER BY original_filename", conn)
    chunks_df = pd.read_sql_query(
        """
        SELECT chunk_id, document_id, chunk_index, chunk_text, section_header, char_count,
               original_filename, document_type, month, project, service
        FROM rag_chunks
        ORDER BY original_filename, chunk_index
        """,
        conn,
    )

# pandas may turn NULL into NaN; keep real None so the metadata logic below is predictable
docs_df = docs_df.astype(object).where(docs_df.notna(), None)
chunks_df = chunks_df.astype(object).where(chunks_df.notna(), None)
chunks_df["char_count"] = chunks_df["char_count"].astype(int)
chunks_df["chunk_index"] = chunks_df["chunk_index"].astype(int)

print(f"{len(docs_df)} documents / {len(chunks_df)} chunks")
assert docs_df["file_path"].is_unique, "Duplicate documents found - run the fixed (replace) ingestion first."
assert set(chunks_df["document_id"]) <= set(docs_df["document_id"]), "Orphan chunks found."

10 documents / 60 chunks


In [3]:
per_doc = chunks_df.groupby("original_filename").agg(
    chunks=("chunk_id", "count"),
    avg_chars=("char_count", "mean"),
    max_chars=("char_count", "max"),
    with_header=("section_header", lambda s: int(s.notna().sum())),
).round(0)
per_doc

,chunks,avg_chars,max_chars,with_header
original_filename,,,,
Task-3.pdf,24,257.0,296,0
email_payload_spec.pdf,7,263.0,296,0
invoice_2024-01_projectAlpha_computeEngine.pdf,3,166.0,202,0
invoice_2024-01_projectAlpha_computeEngine.txt,3,145.0,287,0
invoice_2024-02_projectBeta_cloudStorage.txt,2,193.0,297,0
policy_2024-01_projectAlpha_general.md,3,167.0,222,3
policy_2024-03_projectBeta_billing.md,3,173.0,249,3
report_2024-02_projectAlpha_computeEngine.txt,3,179.0,296,0
report_2024-03_projectBeta_security.txt,3,167.0,278,0


In [4]:
chunks_df[["original_filename", "section_header", "chunk_text"]].head(8)

,original_filename,section_header,chunk_text
0,Task-3.pdf,None,Trainee 3:\n\nConversational FinOps Assistant Specialist\n\n* Main Task:\n\nDevelop an...
1,Task-3.pdf,None,To provide an intuitive interface for users to get real-time cost insights without dee...
2,Task-3.pdf,None,Define Assistant Capabilities (5 hours)\n\n* Goal:\n\nClearly define what questions th...
3,Task-3.pdf,None,"* Steps:\n\n* Interview potential users (e.g., project managers, finance analysts) to ..."
4,Task-3.pdf,None,"* Prioritize common queries and categorize them by type (e.g., cost retrieval, trend a..."
5,Task-3.pdf,None,"* Tools:\n\nStakeholder interviews, Google Docs/Sheets for requirements gathering.\n\n..."
6,Task-3.pdf,None,* Understand the schema and content of the GCP Cloud Billing Export in BigQuery.\n\n* ...
7,Task-3.pdf,None,* Develop optimized SQL queries to retrieve specific cost metrics and dimensions (e.g....


## 2. Embedding basics

With `normalize_embeddings=True` every vector has length 1, so the **dot product equals cosine similarity**, and Chroma's cosine *distance* is `1 - cosine`. The retrieval *score* used everywhere in this project is therefore `1 - distance`.

`max_seq_length` is the model's hard input limit: longer text is **silently truncated**, so what is past the limit is invisible to retrieval.

In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(MODEL_NAME)
print("dimension      :", model.get_sentence_embedding_dimension())
print("max_seq_length :", model.max_seq_length, "tokens")

demo_sentences = [
    "The total amount due is $4,500 USD.",
    "How much do we have to pay?",
    "Passwords must be rotated every 90 days.",
    "المبلغ الإجمالي المستحق هو 4500 دولار",
    "كلمات المرور يجب تغييرها كل 90 يوما",
]
demo_vecs = model.encode(demo_sentences, normalize_embeddings=True)
pd.DataFrame(
    demo_vecs @ demo_vecs.T,
    index=[f"s{i}: {s[:34]}" for i, s in enumerate(demo_sentences)],
    columns=[f"s{i}" for i in range(len(demo_sentences))],
).round(2)

c:\Users\salha\miniconda3\envs\data-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2195.62it/s]


dimension      : 384
max_seq_length : 128 tokens


C:\Users\salha\AppData\Local\Temp\ipykernel_8280\4224402321.py:4: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("dimension      :", model.get_sentence_embedding_dimension())


,s0,s1,s2,s3,s4
"s0: The total amount due is $4,500 USD",1.00,0.36,0.02,0.87,0.07
s1: How much do we have to pay?,0.36,1.00,0.10,0.43,0.11
s2: Passwords must be rotated every 90,0.02,0.10,1.00,0.03,0.31
s3: المبلغ الإجمالي المستحق هو 4500 دو,0.87,0.43,0.03,1.00,0.06
s4: كلمات المرور يجب تغييرها كل 90 يوم,0.07,0.11,0.31,0.06,1.00


**Read the matrix:** s0↔s1 (same idea, different words) should score clearly higher than s0↔s2 (unrelated), and the Arabic pair s3↔s0 shows whether the model really is cross-lingual. These numbers also give a first feeling for what "related" vs "unrelated" looks like on this model's scale — the threshold in section 7 lives on the same scale.

## 3. Two candidate embedding texts + token-limit check

The Week 7 chunker deliberately does **not** repeat `section_header` inside `chunk_text`. For retrieval, the header can carry useful context ("Authentication", "Budget Limits"), so we compare:

- `text_only` → embed `chunk_text`
- `header_plus_text` → embed `section_header + "\n" + chunk_text` (when a header exists)

Whatever is chosen, the **stored / displayed / cited text is always the original `chunk_text`**.

In [6]:
VARIANTS = {
    "text_only": lambda r: r["chunk_text"],
    "header_plus_text": lambda r: (
        f"{r['section_header']}\n{r['chunk_text']}" if r["section_header"] else r["chunk_text"]
    ),
}

records = chunks_df.to_dict("records")
variant_texts = {name: [fn(r) for r in records] for name, fn in VARIANTS.items()}


def n_tokens(text: str) -> int:
    return len(model.tokenizer(text, add_special_tokens=True, truncation=False)["input_ids"])


token_rows, over_limit = [], {}
for name, texts in variant_texts.items():
    lens = np.array([n_tokens(t) for t in texts])
    over_idx = [i for i, n in enumerate(lens) if n > model.max_seq_length]
    over_limit[name] = sorted({records[i]["original_filename"] for i in over_idx})
    token_rows.append({
        "variant": name,
        "mean_tokens": round(float(lens.mean()), 1),
        "max_tokens": int(lens.max()),
        f"chunks_over_{model.max_seq_length}": len(over_idx),
    })
display(pd.DataFrame(token_rows))
print("Files with truncated chunks:", over_limit)

,variant,mean_tokens,max_tokens,chunks_over_128
0,text_only,57.5,86,0
1,header_plus_text,58.0,86,0


Files with truncated chunks: {'text_only': [], 'header_plus_text': []}


## 4. Embed all chunks (both variants)

In [7]:
chunk_vecs = {
    name: model.encode(texts, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
    for name, texts in variant_texts.items()
}
{name: v.shape for name, v in chunk_vecs.items()}

{'text_only': (60, 384), 'header_plus_text': (60, 384)}


## 5. Evaluation questions

The first draft (10 questions) was too small: it showed that retrieval itself is fine, but it could not support a threshold decision. This version has **34 answerable + 15 unanswerable** questions written against the real documents.

- `expected_sources` is a list (any one counts) and supports `*` wildcards (the invoice exists as `.txt` and `.pdf`; the Arabic PDF's real filename may contain spaces or underscores).
- Unanswerable questions have a `kind`, so we can see **which kind of missing information is hard**: `near_topic_missing` (same topic, the fact is simply not there), `unknown_entity` (a project that does not exist), `off_topic`, `filter_no_match` (filters match no document).
- Every question has a `split` (`dev` / `test`, every 3rd question of each class goes to `test`). Rules are **tuned on dev and judged on test**, so the numbers are not just memorised answers.

The JSON is created once. **If you still have the old 10-question file, delete `data/eval/week8/eval_questions.json` and re-run this cell** (the notebook never overwrites it silently).


In [8]:

TXT_A, PDF_A = "invoice_2024-01_projectAlpha_computeEngine.txt", "invoice_2024-01_projectAlpha_computeEngine.pdf"
TXT_B = "invoice_2024-02_projectBeta_cloudStorage.txt"
POL_A, POL_B = "policy_2024-01_projectAlpha_general.md", "policy_2024-03_projectBeta_billing.md"
REP_A, REP_B = "report_2024-02_projectAlpha_computeEngine.txt", "report_2024-03_projectBeta_security.txt"
SPEC, TASK3, AR = "email_payload_spec.pdf", "Task-3.pdf", "الوظيفة*الثانية*.pdf"


def Q(qid, question, expected, tests, filters=None):
    return {"id": qid, "question": question, "filters": filters or {}, "expected_sources": expected,
            "should_reject": False, "kind": "answerable", "tests": tests}


def NEG(qid, question, kind, tests, filters=None):
    return {"id": qid, "question": question, "filters": filters or {}, "expected_sources": [],
            "should_reject": True, "kind": kind, "tests": tests}


DRAFT_QUESTIONS = [
    # ---- answerable (original 8 keep their ids)
    Q(1, "What is the total amount due for Project Alpha in January 2024?", [TXT_A, PDF_A], "two valid sources (txt + pdf)"),
    Q(2, "Which vulnerabilities were found in Project Beta's March security scan?", [REP_B], "report"),
    Q(3, "What is the minimum password length for Project Alpha?", [POL_A], "policy"),
    Q(4, "Who must approve spending that exceeds Project Beta's budget?", [POL_B], "policy"),
    Q(5, "Which fields are required in the email ingestion payload?", [SPEC], "unclassified PDF"),
    Q(6, "What is the unit price of the n2-highmem-8 instance?", [PDF_A], "PDF table"),
    Q(7, "متى موعد إجراء المقابلات؟", [AR], "Arabic"),
    Q(8, "What is the total amount due?", [TXT_B], "metadata filter ($and)", {"project": "projectBeta", "document_type": "invoice"}),
    Q(11, "How many hours of high-memory instances were used for Project Alpha in January 2024?", [TXT_A, PDF_A], "txt + pdf"),
    Q(12, "Within how many days must the Project Alpha invoice be paid?", [TXT_A, PDF_A], "txt + pdf"),
    Q(13, "How much data did the engineering team store for Project Beta in February 2024?", [TXT_B], "invoice"),
    Q(14, "How is payment made for the Project Beta storage invoice?", [TXT_B], "invoice"),
    Q(15, "What is the price per GB of persistent SSD storage?", [PDF_A], "PDF table"),
    Q(16, "How much does network egress cost per GB?", [PDF_A], "PDF table"),
    Q(17, "How often must passwords be rotated for Project Alpha?", [POL_A], "policy"),
    Q(18, "Is direct SSH access allowed from public IP addresses?", [POL_A], "policy"),
    Q(19, "What is required to access the compute engines from outside the corporate network?", [POL_A], "policy"),
    Q(20, "What is the maximum monthly budget for Project Beta's cloud infrastructure?", [POL_B], "policy"),
    Q(21, "By which day of the month must cloud storage invoices be submitted to the finance department?", [POL_B], "policy"),
    Q(22, "What are the consequences of late invoice submissions?", [POL_B], "policy"),
    Q(23, "What uptime did Project Alpha's Compute Engine maintain in February 2024?", [REP_A], "report"),
    Q(24, "What caused the CPU usage spike on February 14th?", [REP_A], "report"),
    Q(25, "What does the report recommend to handle sudden traffic spikes?", [REP_A], "report"),
    Q(26, "When were the outdated SSL certificates rotated?", [REP_B], "report"),
    Q(27, "Is Project Beta compliant with the internal security standards after the scan?", [REP_B], "report"),
    Q(28, "What is the mailbox_id field used for?", [SPEC], "unclassified PDF"),
    Q(29, "What value does html_body currently have in the email payload?", [SPEC], "unclassified PDF"),
    Q(30, "Which GCP role is needed to read the billing export data in BigQuery?", [TASK3], "unclassified PDF"),
    Q(31, "How many hours are estimated for building the chat interface?", [TASK3], "unclassified PDF"),
    Q(32, "What is the estimated total number of hours for the Trainee 3 task?", [TASK3], "unclassified PDF"),
    Q(33, "من الذي يوافق على إضافة دورة جديدة؟", [AR], "Arabic"),
    Q(34, "متى يجب إرسال التقرير المختصر عن الوظيفة؟", [AR], "Arabic"),
    Q(35, "ما المطلوب في بند توزيع الحمل؟", [AR], "Arabic"),
    Q(36, "What is the total amount due?", [TXT_A, PDF_A], "metadata filter (document_type + month)",
      {"document_type": "invoice", "month": "2024-01"}),
    # ---- unanswerable
    NEG(9, "What is the refund policy for Project Beta?", "near_topic_missing", "close to the billing policy, no refund policy exists"),
    NEG(10, "What was the cost of Project Gamma in January 2024?", "unknown_entity", "looks like the invoices, project does not exist"),
    NEG(37, "What is the multi-factor authentication policy for Project Beta?", "near_topic_missing", "MFA exists only for Alpha"),
    NEG(38, "Who is the Chief Technology Officer of CloudTech Solutions?", "near_topic_missing", "the role is mentioned, no name"),
    NEG(39, "What was the total amount due for Project Beta in March 2024?", "near_topic_missing", "Beta's invoice is February"),
    NEG(40, "What was the uptime of Project Alpha's Compute Engine in March 2024?", "near_topic_missing", "uptime exists for February only"),
    NEG(41, "How many vulnerabilities were found in Project Alpha's security scan?", "near_topic_missing", "the scan exists for Beta only"),
    NEG(42, "What is the password policy for Project Delta?", "unknown_entity", "project does not exist"),
    NEG(43, "What is the maximum size of an email attachment in the payload?", "near_topic_missing", "spec lists attachment names only"),
    NEG(44, "How much does Cloud Run cost per hour?", "near_topic_missing", "Cloud Run is mentioned, no pricing"),
    NEG(45, "What is the capital of France?", "off_topic", "unrelated"),
    NEG(46, "How do I bake sourdough bread?", "off_topic", "unrelated"),
    NEG(47, "ما هي سياسة الإجازات السنوية للموظفين؟", "off_topic", "Arabic, unrelated"),
    NEG(48, "كم راتب المدرب في المنصة؟", "near_topic_missing", "Arabic, trainer is mentioned, no salary"),
    NEG(49, "What is the total amount due?", "filter_no_match", "no document matches these filters",
        {"project": "projectBeta", "month": "2024-01"}),
]


def assign_splits(questions):
    seen = {True: 0, False: 0}
    for q in questions:
        k = q["should_reject"]
        q["split"] = "test" if seen[k] % 3 == 2 else "dev"
        seen[k] += 1
    return questions


if EVAL_PATH.exists():
    EVAL = json.loads(EVAL_PATH.read_text(encoding="utf-8"))
    print("Loaded existing", EVAL_PATH)
    if len(EVAL) != len(DRAFT_QUESTIONS):
        print(f"NOTE: the file has {len(EVAL)} questions but draft v2 has {len(DRAFT_QUESTIONS)}. "
              "Delete the JSON and re-run this cell to regenerate it.")
else:
    EVAL_PATH.parent.mkdir(parents=True, exist_ok=True)
    EVAL = assign_splits(DRAFT_QUESTIONS)
    EVAL_PATH.write_text(json.dumps(EVAL, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Created draft v2", EVAL_PATH)
if any("split" not in q for q in EVAL):
    assign_splits(EVAL)

# every expected source must correspond to a real stored document, otherwise a "miss" would be a typo
stored_files = docs_df["original_filename"].tolist()
for q in EVAL:
    for pattern in q["expected_sources"]:
        assert any(fnmatch.fnmatchcase(f, pattern) for f in stored_files), (
            f"Q{q['id']}: no stored document matches {pattern!r}. Stored files: {stored_files}"
        )

n_pos = sum(not q["should_reject"] for q in EVAL)
print(f"{len(EVAL)} questions: {n_pos} answerable / {len(EVAL) - n_pos} unanswerable | "
      f"dev={sum(q['split'] == 'dev' for q in EVAL)} test={sum(q['split'] == 'test' for q in EVAL)}")


Created draft v2 E:\DigitalFutureProjects\TrainingProjects\standalone-data-rag-assistant\data\eval\week8\eval_questions.json
49 questions: 34 answerable / 15 unanswerable | dev=33 test=16


## 6. In-memory retrieval + variant comparison

To compare variants quickly we use plain numpy cosine here. Metadata is built with the exact rule the real `vector_store.py` will use: **NULL values are never stored** — a missing key simply never matches a filter, and `document_type` becomes the explicit value `"unclassified"`.

In [9]:
def to_chroma_metadata(row: dict) -> dict:
    meta = {
        "document_id": row["document_id"],
        "original_filename": row["original_filename"],
        "document_type": row["document_type"] or "unclassified",
    }
    for key in ("month", "project", "service"):
        if row[key]:                      # NULL / empty -> key is simply not stored
            meta[key] = row[key]
    return meta


metas = [to_chroma_metadata(r) for r in records]
q_vecs = model.encode([q["question"] for q in EVAL], normalize_embeddings=True)
K_MAX = 10


def retrieve(q_vec, vecs, k, filters=None):
    """Top-k (index, score). None means: no chunk matches the filter."""
    idx = [i for i, m in enumerate(metas) if all(m.get(f) == v for f, v in (filters or {}).items())]
    if not idx:
        return None
    sims = vecs[idx] @ q_vec
    order = np.argsort(-sims)[:k]
    return [(idx[o], float(sims[o])) for o in order]


def matches_expected(filename, patterns):
    return any(fnmatch.fnmatchcase(filename, p) for p in patterns)


def evaluate(variant):
    vecs = chunk_vecs[variant]
    out = []
    for q, qv in zip(EVAL, q_vecs):
        res = retrieve(qv, vecs, K_MAX, q.get("filters"))
        if res is None:
            out.append({"id": q["id"], "should_reject": q["should_reject"], "status": "no_documents_match_filter",
                        "top1_score": None, "top1_file": None, "rank": None, "top4": []})
            continue
        files = [metas[i]["original_filename"] for i, _ in res]
        scores = [s for _, s in res]
        rank = None
        if not q["should_reject"]:
            rank = next((pos for pos, f in enumerate(files, 1) if matches_expected(f, q["expected_sources"])), None)
        out.append({
            "id": q["id"], "should_reject": q["should_reject"], "status": "retrieved",
            "top1_score": round(scores[0], 3), "top1_file": files[0], "rank": rank,
            "top4": [f"{f[:30]} ({s:.2f})" for f, s in list(zip(files, scores))[:4]],
        })
    df = pd.DataFrame(out)
    df["rank"] = pd.to_numeric(df["rank"])
    df["top1_score"] = pd.to_numeric(df["top1_score"])
    return df


def summarize(df, ks=(1, 3, 4)):
    pos, neg = df[~df["should_reject"]], df[df["should_reject"]]
    res = {f"hit@{k}": float((pos["rank"] <= k).mean()) for k in ks}
    res["mrr"] = float((1 / pos["rank"]).fillna(0).mean())
    res["margin"] = float(pos["top1_score"].min() - neg["top1_score"].max())
    return res


results = {name: evaluate(name) for name in VARIANTS}
comparison = pd.DataFrame({name: summarize(df) for name, df in results.items()}).T.round(3)
comparison

,hit@1,hit@3,hit@4,mrr,margin
text_only,1.0,1.0,1.0,1.0,-0.301
header_plus_text,1.0,1.0,1.0,1.0,-0.301


In [10]:
CHOSEN_VARIANT = comparison.sort_values(["mrr", "margin"], ascending=False).index[0]
# CHOSEN_VARIANT = "text_only"   # <- uncomment to override manually
print("Chosen embedding text variant:", CHOSEN_VARIANT)

detail = results[CHOSEN_VARIANT].merge(
    pd.DataFrame(EVAL)[["id", "question", "tests"]], on="id"
)[["id", "question", "should_reject", "rank", "top1_score", "top1_file", "status"]]
detail

Chosen embedding text variant: text_only


,id,question,should_reject,rank,top1_score,top1_file,status
0,1,What is the total amount due for Project Alpha in January 2024?,False,1.0,0.708,invoice_2024-01_projectAlpha_computeEngine.txt,retrieved
1,2,Which vulnerabilities were found in Project Beta's March security scan?,False,1.0,0.783,report_2024-03_projectBeta_security.txt,retrieved
2,3,What is the minimum password length for Project Alpha?,False,1.0,0.687,policy_2024-01_projectAlpha_general.md,retrieved
3,4,Who must approve spending that exceeds Project Beta's budget?,False,1.0,0.622,policy_2024-03_projectBeta_billing.md,retrieved
4,5,Which fields are required in the email ingestion payload?,False,1.0,0.596,email_payload_spec.pdf,retrieved
5,6,What is the unit price of the n2-highmem-8 instance?,False,1.0,0.554,invoice_2024-01_projectAlpha_computeEngine.pdf,retrieved
6,7,متى موعد إجراء المقابلات؟,False,1.0,0.624,الوظيفة الثانية (1).pdf,retrieved
7,8,What is the total amount due?,False,1.0,0.540,invoice_2024-02_projectBeta_cloudStorage.txt,retrieved
8,11,How many hours of high-memory instances were used for Project Alpha in January 2024?,False,1.0,0.677,invoice_2024-01_projectAlpha_computeEngine.txt,retrieved
9,12,Within how many days must the Project Alpha invoice be paid?,False,1.0,0.766,invoice_2024-01_projectAlpha_computeEngine.txt,retrieved


In [11]:
# what did the top-4 look like for each question? (duplicates txt/pdf of the same invoice show up here)
for _, r in results[CHOSEN_VARIANT].iterrows():
    print(f"Q{r['id']:>2}  rank={r['rank']}  ", " | ".join(r["top4"]))

Q 1  rank=1.0   invoice_2024-01_projectAlpha_c (0.71) | invoice_2024-02_projectBeta_cl (0.64) | policy_2024-03_projectBeta_bil (0.62) | invoice_2024-01_projectAlpha_c (0.57)
Q 2  rank=1.0   report_2024-03_projectBeta_sec (0.78) | report_2024-03_projectBeta_sec (0.76) | invoice_2024-01_projectAlpha_c (0.43) | report_2024-03_projectBeta_sec (0.40)
Q 3  rank=1.0   policy_2024-01_projectAlpha_ge (0.69) | invoice_2024-01_projectAlpha_c (0.54) | policy_2024-03_projectBeta_bil (0.47) | report_2024-02_projectAlpha_co (0.40)
Q 4  rank=1.0   policy_2024-03_projectBeta_bil (0.62) | policy_2024-03_projectBeta_bil (0.62) | policy_2024-03_projectBeta_bil (0.56) | invoice_2024-01_projectAlpha_c (0.51)
Q 5  rank=1.0   email_payload_spec.pdf (0.60) | email_payload_spec.pdf (0.51) | الوظيفة الثانية (1).pdf (0.50) | email_payload_spec.pdf (0.48)
Q 6  rank=1.0   invoice_2024-01_projectAlpha_c (0.55) | invoice_2024-02_projectBeta_cl (0.43) | invoice_2024-01_projectAlpha_c (0.34) | policy_2024-03_projectBet

## 7. Choosing `top_k`

How often is a correct source inside the top-k, for k = 1…6? The smallest k that reaches the maximum is the natural default (never below 3, so the txt/pdf duplicate of the same invoice cannot crowd out a second relevant source).

In [12]:
ks = range(1, 7)
hit_table = pd.DataFrame({
    name: {f"k={k}": float((df[~df["should_reject"]]["rank"] <= k).mean()) for k in ks}
    for name, df in results.items()
}).round(3)
display(hit_table)

best_hit = hit_table[CHOSEN_VARIANT].max()
k_saturation = next(int(label.split("=")[1]) for label, v in hit_table[CHOSEN_VARIANT].items() if v == best_hit)
SUGGESTED_TOP_K = max(k_saturation, 3)
print(f"Hit@k saturates at k={k_saturation} -> suggested RAG_TOP_K = {SUGGESTED_TOP_K}")

,text_only,header_plus_text
k=1,1.0,1.0
k=2,1.0,1.0
k=3,1.0,1.0
k=4,1.0,1.0
k=5,1.0,1.0
k=6,1.0,1.0


Hit@k saturates at k=1 -> suggested RAG_TOP_K = 3



## 8. Can ONE similarity threshold separate answerable from unanswerable questions?

Retrieval always returns *something*. The question here is whether the top-1 cosine score tells us if that something actually **answers** the question. For every threshold `t` we measure the share of answerable questions still accepted (`score >= t`) and the share of unanswerable ones rejected (`score < t`, or no document matches the filters).

A threshold exists only if some `t` gives 100% / 100%. If the two score ranges overlap, no single `t` can — and picking one anyway would be arbitrary.


In [13]:

res = results[CHOSEN_VARIANT]
pos, neg = res[~res["should_reject"]], res[res["should_reject"]]
neg_scored = neg[neg["status"] == "retrieved"]

print("Top-1 score, answerable   : min %.3f | median %.3f | max %.3f" % (pos.top1_score.min(), pos.top1_score.median(), pos.top1_score.max()))
print("Top-1 score, unanswerable : min %.3f | median %.3f | max %.3f" % (neg_scored.top1_score.min(), neg_scored.top1_score.median(), neg_scored.top1_score.max()))
overlap = int((pos["top1_score"] <= neg_scored["top1_score"].max()).sum())
print(f"{overlap}/{len(pos)} answerable questions score at or below the HIGHEST unanswerable one.")
print("Answerable questions whose top-1 is NOT an expected source:", pos[pos["rank"] != 1]["id"].tolist() or "none")

rows = []
for t in np.round(np.arange(0.10, 0.90, 0.02), 2):
    rows.append({
        "threshold": t,
        "answerable_accepted": float((pos["top1_score"] >= t).mean()),
        "unanswerable_rejected": float(((neg["status"] == "no_documents_match_filter") | (neg["top1_score"] < t)).mean()),
    })
sweep = pd.DataFrame(rows)
separable = bool(((sweep["answerable_accepted"] == 1.0) & (sweep["unanswerable_rejected"] == 1.0)).any())
print("\nA single threshold that separates everything exists:", separable)

ok = sweep[sweep["answerable_accepted"] >= 0.90]
if len(ok):
    r = ok.iloc[-1]
    print(f"Highest threshold that still accepts >= 90% of answerable questions: {r.threshold} "
          f"-> it rejects only {r.unanswerable_rejected:.0%} of the unanswerable ones.")
display(sweep.iloc[::3].reset_index(drop=True).round(3))

kinds = res.merge(pd.DataFrame(EVAL)[["id", "kind"]], on="id")
display(kinds.groupby("kind").agg(n=("id", "count"), top1_min=("top1_score", "min"), top1_median=("top1_score", "median"), top1_max=("top1_score", "max")).round(3))


Top-1 score, answerable   : min 0.435 | median 0.659 | max 0.845
Top-1 score, unanswerable : min 0.070 | median 0.584 | max 0.736
28/34 answerable questions score at or below the HIGHEST unanswerable one.
Answerable questions whose top-1 is NOT an expected source: none

A single threshold that separates everything exists: False
Highest threshold that still accepts >= 90% of answerable questions: 0.5 -> it rejects only 40% of the unanswerable ones.


,threshold,answerable_accepted,unanswerable_rejected
0,0.10,1.000,0.133
1,0.16,1.000,0.200
2,0.22,1.000,0.200
3,0.28,1.000,0.200
4,0.34,1.000,0.200
5,0.40,1.000,0.200
6,0.46,0.971,0.200
7,0.52,0.882,0.467
8,0.58,0.735,0.533
9,0.64,0.529,0.733


,n,top1_min,top1_median,top1_max
kind,,,,
answerable,34,0.435,0.659,0.845
filter_no_match,1,NaN,NaN,NaN
near_topic_missing,9,0.468,0.624,0.736
off_topic,3,0.070,0.122,0.501
unknown_entity,2,0.478,0.558,0.637


## 9. Answerability signals: what else separates them?

Cosine similarity measures **topical closeness**, not "does this chunk contain the answer". "What is the refund policy for Project Beta?" is topically very close to Project Beta's billing documents even though no refund policy exists. So we test other signals, each computed for the **top-1 chunk** (the one the extractive answer would use)

In [14]:
import re

EN_STOP = set('''a an the of in on at to for from by with and or is are was were be been being what which who whom
whose when where why how do does did can could should would will shall may might must it its this that these those
there their they them we our you your i me my as if than then so not no into over under about per any all each have
has had'''.split())
AR_STOP = set('متى ما ماذا من كم هل هو هي في على إلى الى عن مع كيف أين لماذا التي الذي ذلك هذا هذه'.split())
STOP = EN_STOP | AR_STOP


def content_stems(text):
    stems = set()
    for tok in re.findall(r'\w+', text.lower()):
        if tok in STOP or len(tok) < 2:
            continue
        if tok.startswith('ال') and len(tok) > 3:
            tok = tok[2:]
        stems.add(tok[:5])
    return stems


def lexical_coverage(question, chunk_text):
    q = content_stems(question)
    return len(q & content_stems(chunk_text)) / len(q) if q else float('nan')


corpus_vocab = {t for r in records for t in re.findall(r'\w+', r['chunk_text'].lower())}


def unknown_entities(question):
    toks = re.findall(r'\w+', question)
    return [t for i, t in enumerate(toks) if i > 0 and t[:1].isupper() and t.lower() not in corpus_vocab]


def auc(pos_vals, neg_vals):
    p = np.asarray(pos_vals, float); n = np.asarray(neg_vals, float)
    p, n = p[~np.isnan(p)], n[~np.isnan(n)]
    if len(p) == 0 or len(n) == 0:
        return float('nan')
    return float((p[:, None] > n[None, :]).mean() + 0.5 * (p[:, None] == n[None, :]).mean())


print("unknown entities, examples:",
      {q['id']: unknown_entities(q['question']) for q in EVAL if q['id'] in (10, 42, 1, 6)})

unknown entities, examples: {1: [], 6: [], 10: ['Gamma'], 42: ['Delta']}


In [15]:
CE_NAME = MODEL_NAME
ce_model = None
try:
    from sentence_transformers import CrossEncoder
    ce_model = CrossEncoder(CE_NAME)
    print('Cross-encoder loaded:', CE_NAME)
except Exception as exc:  # optional signal: never block the notebook
    print('Cross-encoder NOT available -> that signal is skipped:', type(exc).__name__, '-', str(exc)[:160])

Cross-encoder NOT available -> that signal is skipped: RepositoryNotFoundError - 401 Client Error. (Request ID: Root=1-6ac56701-0310a670381d4b2c13e53253;07877068-2470-4bef-93dc-5782ed6e635b)

Repository Not Found for url: https://huggingface


In [16]:
rank_by_id = results[CHOSEN_VARIANT].set_index('id')['rank']
question_by_id = {q['id']: q['question'] for q in EVAL}
candidates, rows = {}, []

for q, qv in zip(EVAL, q_vecs):
    top = retrieve(qv, chunk_vecs[CHOSEN_VARIANT], 3, q.get('filters'))
    base = {'id': q['id'], 'should_reject': q['should_reject'], 'kind': q.get('kind', 'answerable'),
            'split': q['split'], 'rank': rank_by_id[q['id']]}
    if top is None:
        rows.append({**base, 'status': 'no_documents_match_filter', 'top1': np.nan, 'gap': np.nan,
                     'lex': np.nan, 'unknown_ent': 0, 'ce': np.nan})
        continue
    scores = [s for _, s in top]
    candidates[q['id']] = [records[i]['chunk_text'] for i, _ in top]
    rows.append({**base, 'status': 'retrieved', 'top1': scores[0],
                 'gap': scores[0] - scores[1] if len(scores) > 1 else np.nan,
                 'lex': lexical_coverage(q['question'], candidates[q['id']][0]),
                 'unknown_ent': len(unknown_entities(q['question'])), 'ce': np.nan})
sig = pd.DataFrame(rows)

if ce_model is not None:
    for i, row in sig.iterrows():
        if row['status'] == 'retrieved':
            pairs = [(question_by_id[row['id']], t) for t in candidates[row['id']]]
            sig.loc[i, 'ce'] = float(np.max(ce_model.predict(pairs)))

sig.merge(pd.DataFrame(EVAL)[['id', 'question']], on='id')[
    ['id', 'kind', 'split', 'top1', 'gap', 'lex', 'unknown_ent', 'ce', 'question']].round(3).head(60)

,id,kind,split,top1,gap,lex,unknown_ent,ce,question
0,1,answerable,dev,0.708,0.067,1.000,0,NaN,What is the total amount due for Project Alpha in January 2024?
1,2,answerable,dev,0.783,0.022,0.857,0,NaN,Which vulnerabilities were found in Project Beta's March security scan?
2,3,answerable,test,0.687,0.142,0.600,0,NaN,What is the minimum password length for Project Alpha?
3,4,answerable,dev,0.622,0.000,0.833,0,NaN,Who must approve spending that exceeds Project Beta's budget?
4,5,answerable,dev,0.596,0.089,0.600,0,NaN,Which fields are required in the email ingestion payload?
5,6,answerable,test,0.554,0.119,1.000,0,NaN,What is the unit price of the n2-highmem-8 instance?
6,7,answerable,dev,0.624,0.124,0.333,0,NaN,متى موعد إجراء المقابلات؟
7,8,answerable,dev,0.540,0.437,1.000,0,NaN,What is the total amount due?
8,11,answerable,test,0.677,0.059,0.800,0,NaN,How many hours of high-memory instances were used for Project Alpha in January 2024?
9,12,answerable,dev,0.766,0.155,0.714,0,NaN,Within how many days must the Project Alpha invoice be paid?


In [17]:
scored = sig[sig['status'] == 'retrieved']
pos_s, neg_s = scored[~scored['should_reject']], scored[scored['should_reject']]

rows = []
for col, label in [('top1', 'top-1 cosine'), ('gap', 'gap top1-top2'), ('lex', 'lexical coverage'), ('ce', 'cross-encoder')]:
    if scored[col].notna().sum() == 0:
        continue
    rows.append({'signal': label, 'AUC': round(auc(pos_s[col], neg_s[col]), 3),
                 'answerable_median': round(pos_s[col].median(), 3), 'unanswerable_median': round(neg_s[col].median(), 3)})
display(pd.DataFrame(rows))

print('Unknown-entity flag -> unanswerable flagged: %d/%d | answerable wrongly flagged: %d/%d' % (
    int((neg_s['unknown_ent'] > 0).sum()), len(neg_s), int((pos_s['unknown_ent'] > 0).sum()), len(pos_s)))

display(scored.groupby('kind').agg(
    n=('id', 'count'), top1_median=('top1', 'median'), lex_median=('lex', 'median'),
    ce_median=('ce', 'median'), entity_flagged=('unknown_ent', lambda s: int((s > 0).sum()))).round(3))

,signal,AUC,answerable_median,unanswerable_median
0,top-1 cosine,0.672,0.659,0.584
1,gap top1-top2,0.725,0.122,0.046
2,lexical coverage,0.880,0.833,0.450


Unknown-entity flag -> unanswerable flagged: 4/14 | answerable wrongly flagged: 0/34


,n,top1_median,lex_median,ce_median,entity_flagged
kind,,,,,
answerable,34,0.659,0.833,NaN,0
near_topic_missing,9,0.624,0.571,NaN,0
off_topic,3,0.122,0.000,NaN,2
unknown_entity,2,0.557,0.450,NaN,2


### Rule search: tune on `dev`, judge on `test`

A rule **rejects** a question when any condition fires: no document matches the filters · `top1 < t1` · `lex < tl` · unknown entity (if enabled) · `ce < tc`. Candidate thresholds are midpoints between observed values.

Selection is **recall-first**: only rules that still accept at least `POS_FLOOR` (90%) of answerable `dev` questions are eligible (wrongly refusing a valid question is the worse error), then the one rejecting the most unanswerable questions wins; ties go to the simpler rule. The same search restricted to `top1` alone is the baseline, so the table shows what the extra signals really add.

In [18]:
POS_FLOOR = 0.90


def _arrays(df):
    return dict(
        pos=(~df['should_reject']).to_numpy(), nomatch=(df['status'] == 'no_documents_match_filter').to_numpy(),
        top1=df['top1'].to_numpy(float), lex=df['lex'].to_numpy(float), ent=(df['unknown_ent'].to_numpy() > 0),
        ce=df['ce'].to_numpy(float), rank1=(df['rank'].to_numpy(float) == 1))


def _midpoints(values):
    vals = sorted({float(v) for v in values if not np.isnan(v)})
    return [round((a + b) / 2, 3) for a, b in zip(vals, vals[1:])]


def _reject(a, rule):
    rej = a['nomatch'] | (a['top1'] < rule['t1']) | (a['lex'] < rule['tl']) | (a['ce'] < rule['tc'])
    return rej | a['ent'] if rule['entity'] else rej


def rule_search(df, use_lex=True, use_entity=True, use_ce=True):
    a = _arrays(df)
    pos, neg = a['pos'], ~a['pos']
    t1_grid = [0.0] + _midpoints(a['top1'])
    tl_grid = ([0.0] + [round(x, 1) for x in np.arange(0.1, 1.01, 0.1)]) if use_lex else [0.0]
    ent_grid = [False, True] if use_entity else [False]
    tc_grid = [-np.inf] + (_midpoints(a['ce']) if use_ce and np.isfinite(a['ce']).any() else [])
    best = None
    for t1 in t1_grid:
        for tl in tl_grid:
            for ent_on in ent_grid:
                for tc in tc_grid:
                    rule = {'t1': t1, 'tl': tl, 'entity': ent_on, 'tc': tc}
                    rej = _reject(a, rule)
                    pos_acc = float((~rej & pos).sum() / pos.sum())
                    if pos_acc < POS_FLOOR:
                        continue
                    neg_rej = float((rej & neg).sum() / neg.sum())
                    n_active = int(t1 > 0) + int(tl > 0) + int(ent_on) + int(tc > -np.inf)
                    key = (neg_rej, pos_acc, -n_active, -t1, -tl)
                    if best is None or key > best[0]:
                        best = (key, rule)
    return best[1]


def report_rule(df, rule, name):
    a = _arrays(df)
    rej = _reject(a, rule)
    pos = a['pos']
    return {'set': name, 'n_answerable': int(pos.sum()), 'n_unanswerable': int((~pos).sum()),
            'answerable_accepted': round(float((~rej & pos).sum() / pos.sum()), 3),
            'answerable_correct_top1': round(float((~rej & pos & a['rank1']).sum() / pos.sum()), 3),
            'unanswerable_rejected': round(float((rej & ~pos).sum() / (~pos).sum()), 3)}


dev, test = sig[sig['split'] == 'dev'], sig[sig['split'] == 'test']
rule_single = rule_search(dev, use_lex=False, use_entity=False, use_ce=False)
rule_multi = rule_search(dev)
print('top1-only rule :', rule_single)
print('combined rule  :', rule_multi)
display(pd.DataFrame([
    report_rule(dev, rule_single, 'dev  | top1 only'), report_rule(test, rule_single, 'TEST | top1 only'),
    report_rule(dev, rule_multi, 'dev  | combined'), report_rule(test, rule_multi, 'TEST | combined'),
]))

top1-only rule : {'t1': 0.479, 'tl': 0.0, 'entity': False, 'tc': -inf}
combined rule  : {'t1': 0.0, 'tl': np.float64(0.5), 'entity': True, 'tc': -inf}


,set,n_answerable,n_unanswerable,answerable_accepted,answerable_correct_top1,unanswerable_rejected
0,dev | top1 only,23,10,0.957,0.957,0.3
1,TEST | top1 only,11,5,1.000,1.000,0.4
2,dev | combined,23,10,0.957,0.957,0.7
3,TEST | combined,11,5,1.000,1.000,0.4


In [19]:
# which test questions does the combined rule get wrong?
def show_errors(df, rule, title):
    rej = _reject(_arrays(df), rule)
    d = df.assign(rejected=rej).merge(pd.DataFrame(EVAL)[['id', 'question']], on='id')
    wrong = d[(d['should_reject'] & ~d['rejected']) | (~d['should_reject'] & d['rejected'])]
    print(f'{title}: {len(wrong)} wrong decision(s) out of {len(d)}')
    if len(wrong):
        wrong = wrong.assign(error=np.where(wrong['should_reject'], 'LEAK (unanswerable accepted)', 'FALSE REFUSAL'))
        display(wrong[['id', 'kind', 'error', 'top1', 'lex', 'unknown_ent', 'ce', 'question']].round(3))


show_errors(test, rule_multi, 'TEST, combined rule')
show_errors(dev, rule_multi, 'dev,  combined rule')

TEST, combined rule: 3 wrong decision(s) out of 16


,id,kind,error,top1,lex,unknown_ent,ce,question
11,37,near_topic_missing,LEAK (unanswerable accepted),0.561,0.667,0,NaN,What is the multi-factor authentication policy for Project Beta?
12,40,near_topic_missing,LEAK (unanswerable accepted),0.736,0.714,0,NaN,What was the uptime of Project Alpha's Compute Engine in March 2024?
13,43,near_topic_missing,LEAK (unanswerable accepted),0.492,0.600,0,NaN,What is the maximum size of an email attachment in the payload?


dev,  combined rule: 4 wrong decision(s) out of 33


,id,kind,error,top1,lex,unknown_ent,ce,question
4,7,answerable,FALSE REFUSAL,0.624,0.333,0,NaN,متى موعد إجراء المقابلات؟
23,9,near_topic_missing,LEAK (unanswerable accepted),0.606,0.500,0,NaN,What is the refund policy for Project Beta?
26,39,near_topic_missing,LEAK (unanswerable accepted),0.691,0.857,0,NaN,What was the total amount due for Project Beta in March 2024?
27,41,near_topic_missing,LEAK (unanswerable accepted),0.705,0.571,0,NaN,How many vulnerabilities were found in Project Alpha's security scan?


In [20]:
# final constants: refit on ALL questions (only meaningful if the TEST table above looked acceptable)
FINAL_RULE = rule_search(sig)
display(pd.DataFrame([report_rule(sig, FINAL_RULE, 'ALL (refit)')]))
print('FINAL_RULE =', FINAL_RULE)
print('\nIf TEST is much worse than dev, the rule is overfitted: grow the question set before trusting it.')
print('Whatever is left unrejected is NOT a bug in the rule: it is the part that must be handled by the')
print('grounded prompt + LLM in Week 9, or flagged to the user as low confidence.')

,set,n_answerable,n_unanswerable,answerable_accepted,answerable_correct_top1,unanswerable_rejected
0,ALL (refit),34,15,0.971,0.971,0.6


FINAL_RULE = {'t1': 0.0, 'tl': np.float64(0.5), 'entity': True, 'tc': -inf}

If TEST is much worse than dev, the rule is overfitted: grow the question set before trusting it.
Whatever is left unrejected is NOT a bug in the rule: it is the part that must be handled by the
grounded prompt + LLM in Week 9, or flagged to the user as low confidence.


## 10. Arabic check (question 7)

Arabic text extracted from PDFs can carry two artifacts: (1) **missing spaces** between words (already documented in Week 7), and (2) possibly a **lost alef after lam** (the "لا" ligature: `المقابالت` instead of `المقابلات`) depending on the PDF and the extractor. This cell first **detects** whether artifact (2) is really present in *your* stored chunks, then measures retrieval with (a) the correct spelling, (b) the spelling as a broken extraction would write it, and (c) an English question against the Arabic document.

In [21]:
ar_mask = docs_df["original_filename"].str.contains("الوظيفة")
print("Arabic documents found:", docs_df.loc[ar_mask, "original_filename"].tolist())
ar_text = " ".join(r["chunk_text"] for r in records if "الوظيفة" in r["original_filename"])
print("Lost-alef ligature artifact present in stored text:", any(w in ar_text for w in ("المقابالت", "خالل", "األحد")))
print("(if False, row (b) below is just a spelling-robustness test, not an extraction-artifact test)")
for r in records:
    if "الوظيفة" in r["original_filename"]:
        print("-" * 60)
        print(r["chunk_text"][:300])

arabic_queries = {
    "correct spelling": "متى موعد إجراء المقابلات؟",
    "as extracted (broken ligature)": "متى موعد إجراء المقابالت؟",
    "english question": "When are the interviews held?",
}
vecs = chunk_vecs[CHOSEN_VARIANT]
rows = []
for label, text in arabic_queries.items():
    qv = model.encode([text], normalize_embeddings=True)[0]
    top = retrieve(qv, vecs, 3)
    rows.append({
        "query": label,
        "top1_file": metas[top[0][0]]["original_filename"][:35],
        "top1_score": round(top[0][1], 3),
        "expected_in_top3": any(matches_expected(metas[i]["original_filename"], [AR]) for i, _ in top),
    })
pd.DataFrame(rows)

Arabic documents found: ['الوظيفة الثانية (1).pdf']
Lost-alef ligature artifact present in stored text: False
(if False, row (b) below is just a spelling-robustness test, not an extraction-artifact test)
------------------------------------------------------------
الوظيفة الثانية نريد تطوير نظام تعليم الكتروني يهدف إلى تسهيل عملية التعليم من خلال إدارة المستخدمين (مسؤولين، مدربين، متعلمين)، نشر الدورات التعليمية، إدارة عمليات الاشتراك والدفع، والتقييمات والاختبارات: يمكن للمسؤول admin إضافةمدربين، بحيث يتمكن المدرب من تسجيل الدخول للنظام وإضافة دورات تابعة
------------------------------------------------------------
المدرب من تسجيل الدخول للنظام وإضافة دورات تابعة له وإدارتها، يقوم المدرب بإنشاء اختبار لكل دورة لتقييم المشتركين وتتبع حالتهم.
------------------------------------------------------------
اختبار لكل دورة لتقييم المشتركين وتتبع حالتهم. يجب أن تتم الموافقة من قبل الـ admin عند إضافة دورة جديدة يمكن للمستخدمين التسجيل في النظام واستعراض الدورات المتاحة والاشتراك بها بعد دفع ا

,query,top1_file,top1_score,expected_in_top3
0,correct spelling,الوظيفة الثانية (1).pdf,0.624,True
1,as extracted (broken ligature),الوظيفة الثانية (1).pdf,0.465,True
2,english question,الوظيفة الثانية (1).pdf,0.598,True


## 11. ChromaDB behaviour with our data (temp folder only)

Checks, in order: NULL metadata · load · cosine = numpy · filters (`$eq`, `$and`, no match) · `unclassified` handling · idempotent upsert · the **sync** pattern the indexer will use (the ingestion *replace* gives every chunk a new `chunk_id` on each run, so the index must delete stale ids and add new ones).

In [22]:
import chromadb

tmp_dir = tempfile.mkdtemp(prefix="chroma_exp_")
client = chromadb.PersistentClient(path=tmp_dir)
col = client.create_collection(
    "rag_experiment",
    metadata={"hnsw:space": "cosine", "embedding_model": MODEL_NAME},
)
vecs = chunk_vecs[CHOSEN_VARIANT]
ids = chunks_df["chunk_id"].tolist()
id_to_idx = {cid: i for i, cid in enumerate(ids)}

# (a) what does YOUR installed Chroma do with a None metadata value?
try:
    col.add(ids=["probe"], embeddings=[vecs[0].tolist()], metadatas=[{"document_id": "x", "month": None}])
    print("None ACCEPTED ->", col.get(ids=["probe"])["metadatas"])
    col.delete(ids=["probe"])
except Exception as exc:
    print("None REJECTED ->", type(exc).__name__, "-", exc)
print("(either way we never store None: missing key = never matches a filter)")

None REJECTED -> TypeError - argument 'metadatas': Cannot convert Python object to MetadataValue
(either way we never store None: missing key = never matches a filter)


In [23]:
# (b) load everything with normalized metadata. We always pass embeddings ourselves,
#     so Chroma never needs (or downloads) its own default embedding model.
col.upsert(ids=ids, embeddings=vecs.tolist(), metadatas=metas)
assert col.count() == len(chunks_df)
print("count:", col.count())

# (c) Chroma cosine distance must equal 1 - numpy cosine
qv = q_vecs[0]
hit = col.query(query_embeddings=[qv.tolist()], n_results=3)
chroma_scores = [1 - d for d in hit["distances"][0]]
numpy_scores = [s for _, s in retrieve(qv, vecs, 3)]
print("chroma:", np.round(chroma_scores, 4), "numpy:", np.round(numpy_scores, 4))
assert np.allclose(chroma_scores, numpy_scores, atol=1e-3)

count: 60
chroma: [0.7075 0.6404 0.6225] numpy: [0.7075 0.6404 0.6225]


In [24]:
# (d) filters
def build_where(filters):
    if not filters:
        return None
    items = [{k: {"$eq": v}} for k, v in filters.items()]
    return items[0] if len(items) == 1 else {"$and": items}


q8 = next(q for q in EVAL if q["id"] == 8)
r8 = col.query(query_embeddings=[q_vecs[EVAL.index(q8)].tolist()], n_results=4, where=build_where(q8["filters"]))
print("Q8 filtered files :", sorted({m["original_filename"] for m in r8["metadatas"][0]}))
assert all(m["project"] == "projectBeta" and m["document_type"] == "invoice" for m in r8["metadatas"][0])

# a filter that matches nothing: detect it BEFORE querying, so we can answer 'no_documents_match_filter'
no_match = build_where({"project": "projectGamma"})
print("get(where=no match) ->", col.get(where=no_match, limit=1)["ids"], "(empty = no_documents_match_filter)")
print("query(where=no match) ->", col.query(query_embeddings=[qv.tolist()], n_results=3, where=no_match)["ids"])

Q8 filtered files : ['invoice_2024-02_projectBeta_cloudStorage.txt']
get(where=no match) -> [] (empty = no_documents_match_filter)
query(where=no match) -> [[]]


In [25]:
# (e) unclassified documents
uncl = col.get(where=build_where({"document_type": "unclassified"}), include=["metadatas"])
print("unclassified chunks:", len(uncl["ids"]), "from", sorted({m["original_filename"] for m in uncl["metadatas"]}))

by_month = col.get(where=build_where({"month": "2024-01"}), include=["metadatas"])
assert all(m["document_type"] != "unclassified" for m in by_month["metadatas"])
print("month filter excludes unclassified docs (they have no 'month' key):", True)

# (f) idempotent upsert
col.upsert(ids=ids, embeddings=vecs.tolist(), metadatas=metas)
assert col.count() == len(chunks_df)
print("re-upsert keeps count at", col.count())

unclassified chunks: 40 from ['Task-3.pdf', 'email_payload_spec.pdf', 'الوظيفة الثانية (1).pdf']
month filter excludes unclassified docs (they have no 'month' key): True
re-upsert keeps count at 60


In [26]:
# (g) the sync pattern: simulate drift, then repair it by set difference
col.delete(ids=ids[:3])                                                   # 3 chunks missing from the index
col.upsert(ids=["stale-1", "stale-2"], embeddings=[vecs[0].tolist()] * 2,  # 2 chunks that no longer exist in SQLite
           metadatas=[metas[0], metas[0]])

current_ids = set(ids)
existing_ids = set(col.get(include=[])["ids"])
to_add, to_delete = current_ids - existing_ids, existing_ids - current_ids
print("to_add:", len(to_add), "| to_delete:", len(to_delete))

if to_delete:
    col.delete(ids=list(to_delete))
if to_add:
    add_ids = sorted(to_add, key=id_to_idx.get)
    col.upsert(ids=add_ids, embeddings=[vecs[id_to_idx[i]].tolist() for i in add_ids],
               metadatas=[metas[id_to_idx[i]] for i in add_ids])
assert set(col.get(include=[])["ids"]) == current_ids
print("after sync: index == SQLite ->", col.count(), "chunks")

del col, client
shutil.rmtree(tmp_dir, ignore_errors=True)

to_add: 3 | to_delete: 2
after sync: index == SQLite -> 60 chunks


## 12. Findings → inputs for Phase 2

In [27]:

summary = {
    "model": MODEL_NAME,
    "dimension": getattr(model, "get_embedding_dimension", model.get_sentence_embedding_dimension)(),
    "max_seq_length": model.max_seq_length,
    "embedding_text_variant": CHOSEN_VARIANT,
    "files_with_truncated_chunks": over_limit[CHOSEN_VARIANT],
    "suggested_top_k": SUGGESTED_TOP_K,
    "single_threshold_separates_everything": separable,
    "final_rule": {k: (None if v == -np.inf else v) for k, v in FINAL_RULE.items()},
    "test_report_combined": report_rule(test, rule_multi, "TEST | combined"),
    "test_report_top1_only": report_rule(test, rule_single, "TEST | top1 only"),
    "variant_comparison": comparison.to_dict("index"),
}
print(json.dumps(summary, ensure_ascii=False, indent=2, default=float))

print("\n# -> .env (only the rows with a value are active gates)")
print(f'EMBEDDING_MODEL_NAME="{MODEL_NAME}"')
print('CHROMA_PATH="data/db/chroma"')
print('RAG_COLLECTION="rag_chunks_v1"')
print(f"RAG_TOP_K={SUGGESTED_TOP_K}")
print(f"RAG_MIN_SCORE={FINAL_RULE['t1']}")
print(f"RAG_MIN_LEXICAL_COVERAGE={FINAL_RULE['tl']}")
print(f"RAG_UNKNOWN_ENTITY_CHECK={FINAL_RULE['entity']}")
print(f"RAG_MIN_CROSS_ENCODER_SCORE={'' if FINAL_RULE['tc'] == -np.inf else FINAL_RULE['tc']}")


{
  "model": "paraphrase-multilingual-MiniLM-L12-v2",
  "dimension": 384,
  "max_seq_length": 128,
  "embedding_text_variant": "text_only",
  "files_with_truncated_chunks": [],
  "suggested_top_k": 3,
  "single_threshold_separates_everything": false,
  "final_rule": {
    "t1": 0.0,
    "tl": 0.5,
    "entity": true,
    "tc": null
  },
  "test_report_combined": {
    "set": "TEST | combined",
    "n_answerable": 11,
    "n_unanswerable": 5,
    "answerable_accepted": 1.0,
    "answerable_correct_top1": 1.0,
    "unanswerable_rejected": 0.4
  },
  "test_report_top1_only": {
    "set": "TEST | top1 only",
    "n_answerable": 11,
    "n_unanswerable": 5,
    "answerable_accepted": 1.0,
    "answerable_correct_top1": 1.0,
    "unanswerable_rejected": 0.4
  },
  "variant_comparison": {
    "text_only": {
      "hit@1": 1.0,
      "hit@3": 1.0,
      "hit@4": 1.0,
      "mrr": 1.0,
      "margin": -0.301
    },
    "header_plus_text": {
      "hit@1": 1.0,
      "hit@3": 1.0,
      "hit@4":

### Carry into Phase 2 (`src/rag/week8/`)

- `embedder.py`: encode with `normalize_embeddings=True`; build the embedding text with the **chosen variant**; keep storing/citing the original `chunk_text`.
- `vector_store.py`: collection created with `{"hnsw:space": "cosine", "embedding_model": <name>}` and **always** given explicit embeddings; metadata through `to_chroma_metadata` (never `None`); collection name ≥ 3 characters (`rag_chunks_v1` is fine).
- `indexer.py`: sync by set difference (section 10-g) — `chunk_id`s change on every ingestion.
- `retriever.py`: `build_where` as above; check `get(where=…, limit=1)` first to return `no_documents_match_filter`.
- `answerability.py` (new, only if section 9 justifies it): the rule found there, applied after retrieval; its decision and every signal value are returned in the response (`debug`) so a refusal is always explainable.
- Record in `rag_design_walkthrough.txt`: the variant comparison, truncation findings, the threshold plateau, the Arabic result, and the 10-question caveat.